# Phase 6: Modeling

GitHub issue: #7. Depends on #6 (Phase 5: baseline = 0.0335 NDCG@10). Scope trimmed per
`_doc/decisions/D2-phase6-scope.md`: one learned model (LightGBM), not the full old five-method
comparison (BM25, Two-Tower, DCN-V2+MMoE, Bedrock roles, position-bias study - all out of scope for
Level 0, see that decision for why).

## Step 1: Candidate algorithm

LightGBM `lambdarank` (`LGBMRanker`) - the one model D2 committed to. Tree-based rerankers are the
standard choice here: heterogeneous features, no careful scaling needed, native support for graded
relevance and missing values, and far cheaper than the dropped neural options - matches AGENTS.md's
cost-minimization rule.

In [1]:
from pathlib import Path
import sys

import lightgbm as lgb
import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
FROZEN_DIR = ROOT / "data" / "frozen"

from ranking.eval.bootstrap import bootstrap_ci, paired_bootstrap_pvalue
from ranking.eval.metrics import ndcg_at_k

FEATURE_COLS = ["score", "item_popularity", "prefix_len", "category_match", "category", "query_category"]
CATEGORICAL_COLS = ["category", "query_category"]
SEED = 42

def load(split):
    features = pl.read_parquet(FROZEN_DIR / "features" / f"{split}.parquet")
    prefix_len = pl.read_parquet(FROZEN_DIR / "ranking_examples.parquet").filter(
        pl.col("split") == split
    ).select("session", "prefix_len")
    return (
        features.join(prefix_len, on="session", how="left")
        .with_columns(pl.col("category_match").cast(pl.Int8))
        .sort(["session", "rank"])
    )

def group_sizes(df):
    return df.group_by("session", maintain_order=True).agg(pl.len()).select("len").to_series().to_list()

train = load("train")
val = load("val")
train_groups = group_sizes(train)
val_groups = group_sizes(val)
print(f"train: {train.height:,} rows, {len(train_groups):,} groups")
print(f"val: {val.height:,} rows, {len(val_groups):,} groups")

train: 23,447,741 rows, 487,647 groups
val: 18,266,086 rows, 375,200 groups


## Step 2-3: Training and tuning

Fixed seed 42, 200 estimators with early stopping on val NDCG@10, learning rate 0.05, modest
`num_leaves=31` / `min_child_samples=50` - a single reasonable config, not a tuning grid, matching the
trimmed scope rather than the archived plan's full hyperparameter sweep. Categorical features:
`category`, `query_category` (LightGBM converts our `-1` "unknown" sentinel to NaN for these - its own
convention - and still routes missing values natively during tree splits, so the signal isn't lost,
just implemented differently than intended).

In [2]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    eval_at=[10],
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    random_state=SEED,
    verbosity=-1,
)
ranker.fit(
    train.select(FEATURE_COLS).to_pandas(),
    train["grade"].to_pandas(),
    group=train_groups,
    eval_set=[(val.select(FEATURE_COLS).to_pandas(), val["grade"].to_pandas())],
    eval_group=[val_groups],
    eval_at=[10],
    categorical_feature=CATEGORICAL_COLS,
    callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(0)],
)
print(f"best iteration: {ranker.best_iteration_}")

/Users/shaliqshukoor/search_project/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/Users/shaliqshukoor/search_project/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


best iteration: 198


## Step 4: Imbalance

Not handled with a separate resampling/weighting step. Graded gains (click=1/cart=2/order=3) already
weight the rare, valuable classes higher in the lambdarank loss itself - the standard way LTR models
absorb this kind of imbalance, matching the Phase 3 EDA note that action was deferred to here.

## Step 5: Selection vs. baseline

Same val examples, same candidate pools, same NDCG@10 code as Phase 5 - only the ranking order changes
(raw `score` vs. LightGBM's `pred`), so the comparison is exactly paired per session.

In [3]:
val = val.with_columns(pl.Series("pred", ranker.predict(val.select(FEATURE_COLS).to_pandas())))
val_examples = pl.read_parquet(FROZEN_DIR / "ranking_examples.parquet").filter(pl.col("split") == "val")

def per_session_ndcg(df, examples, score_col):
    ranked = df.sort(["session", score_col], descending=[False, True])
    per_session = ranked.group_by("session", maintain_order=True).agg(pl.col("grade").alias("gains"))
    joined = examples.select("session").join(per_session, on="session", how="left")
    return [ndcg_at_k(g, k=10) if g is not None else 0.0 for g in joined["gains"].to_list()]

baseline_ndcg = per_session_ndcg(val, val_examples, "score")
reranker_ndcg = per_session_ndcg(val, val_examples, "pred")

b_mean, b_lo, b_hi = bootstrap_ci(baseline_ndcg, seed=0)
r_mean, r_lo, r_hi = bootstrap_ci(reranker_ndcg, seed=0)
diff, p_value = paired_bootstrap_pvalue(reranker_ndcg, baseline_ndcg, seed=0)

print(f"baseline (score-ranked) NDCG@10: {b_mean:.4f} [{b_lo:.4f}, {b_hi:.4f}]")
print(f"reranker (LightGBM)     NDCG@10: {r_mean:.4f} [{r_lo:.4f}, {r_hi:.4f}]")
print(f"paired diff (reranker - baseline): {diff:.4f}, p={p_value:.4f}")

/Users/shaliqshukoor/search_project/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


baseline (score-ranked) NDCG@10: 0.0335 [0.0330, 0.0339]
reranker (LightGBM)     NDCG@10: 0.0577 [0.0571, 0.0584]
paired diff (reranker - baseline): 0.0243, p=0.0000


In [4]:
importance = dict(zip(FEATURE_COLS, [int(x) for x in ranker.feature_importances_]))
importance

{'score': 1590,
 'item_popularity': 873,
 'prefix_len': 878,
 'category_match': 211,
 'category': 1358,
 'query_category': 1030}

## Result

| method | NDCG@10 | 95% CI |
|---|---|---|
| baseline (co-visitation score) | 0.0335 | [0.0330, 0.0339] |
| reranker (LightGBM) | **0.0577** | [0.0571, 0.0584] |

Paired bootstrap: mean diff = +0.0243, p < 0.0001 (10,000 resamples) - the reranker wins decisively,
not a tie. Feature importance shows `score` remains the single strongest signal, but `category` and
`query_category` clearly add real information on top of it, which is exactly what a reranker is
supposed to do versus the pool's own raw ordering.

## Options considered

- Tuning `score`-only vs. the full feature set: not run as a separate ablation given the trimmed scope
  - the feature importance breakdown above already shows every feature getting meaningful split usage,
  without a full ablation study.
- BM25, Two-Tower, DCN-V2+MMoE: out of scope per D2.

## Exit criteria

Chosen model (LightGBM lambdarank), config (above), saved artifact, validation metrics vs. baseline
(table above). Met - model saved to `data/frozen/reranker_lgbm.txt` by `scripts/train_reranker.py`
(same logic as this notebook, packaged for unattended reruns).

## Out of scope / follow-ups

- Hyperparameter tuning beyond the single config used here, if Phase 7's test result suggests real
  headroom.
- Whether the LightGBM categorical NaN-conversion behavior for unknown categories meaningfully hurts
  the ~17% of examples with `query_category = -1`.